# Benchmark test-structure check\n\nDraws each abl1 mock-up ground truth and the exact task prompt every agent receives (identical across arms). Run top to bottom in the `chemistree` env.

In [ ]:
import sys
from pathlib import Path

REPO = Path.cwd()
while not (REPO / "benchmarks").exists() and REPO != REPO.parent:
    REPO = REPO.parent
sys.path.insert(0, str(REPO))

In [ ]:
import json

from IPython.display import display
from rdkit import Chem
from rdkit.Chem import Draw

from benchmarks import run
from benchmarks.tasks import strip_murcko


def load(name):
    p = REPO / "benchmarks" / "cases" / name
    return [json.loads(x) for x in p.read_text().splitlines() if x.strip()]


def grid(smis, legends, size=(340, 260)):
    mols = [Chem.MolFromSmiles(s) if isinstance(s, str) else s for s in smis]
    return Draw.MolsToGridImage(
        mols, legends=legends, molsPerRow=len(mols), subImgSize=size
    )

In [ ]:
# Track A -- 2D editing: starting molecule -> gold product + the identical task prompt.
for item in load("abl1_2d.jsonl"):
    print("=" * 100)
    print(f"[{item['category']}] {item['id']}")
    print("INSTRUCTION:", item["instruction"])
    print("\nTASK PROMPT (identical across arms):")
    print(run.build_prompt(item, item["smiles"]))
    display(grid([item["smiles"], item["gold"]], ["start", "gold"]))

In [ ]:
# Track B1 -- scaffold decoration: crystal ligand (target) + Murcko scaffold (seed).
for item in load("abl1_3d_decoration.jsonl"):
    crystal = Chem.MolToSmiles(Chem.MolFromMolFile(str(REPO / item["target"])))
    scaffold = Chem.MolToSmiles(strip_murcko.scaffold_pose(str(REPO / item["target"])))
    print("=" * 100)
    print(f"[decoration] {item['id']}")
    print("INSTRUCTION:", item["instruction"])
    print("\nTASK PROMPT (identical across arms):")
    print(run.build_prompt(item, scaffold))
    display(grid([crystal, scaffold], ["crystal (target)", "scaffold (seed)"]))

In [ ]:
# Track B2 -- understanding probes: posed ligand + question + ground-truth answer.
for item in load("abl1_3d_probes.jsonl"):
    lig = Chem.MolToSmiles(Chem.MolFromMolFile(str(REPO / item["ligand"])))
    print("=" * 100)
    print(f"[probe] {item['id']}")
    print("QUESTION:", item["question"])
    print("GROUND-TRUTH ANSWER:", item["answer"])
    print("\nTASK PROMPT (identical across arms):")
    print(run.build_prompt(item, lig))
    display(grid([lig], ["posed ligand"], size=(520, 380)))